# 📖 Notebook 1: Geospatial Matching

When a rider requests a ride, we need to find the **closest available drivers** fast. This is a proximity search — given a point on Earth, find all points within X kilometers.

Regular database indexes (B-trees) are terrible at this because latitude and longitude are **two dimensions**. Searching both at once requires a full table scan or ugly range queries.

This notebook shows two solutions:
1. **PostGIS** — a PostgreSQL extension with spatial indexes (R-trees) built for this
2. **Redis Geo** — an in-memory geospatial index for ultra-fast lookups

## Learning Objectives

By the end of this notebook, you'll understand:
- Why regular indexes fail for location queries
- How PostGIS uses spatial indexes to find nearby points efficiently
- How Redis Geo uses geohashing for in-memory proximity searches
- When to use each approach (and why Uber uses both)

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/uber
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `uber_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "uber_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 The Problem: Finding Nearby Drivers

Imagine you're standing in downtown San Francisco and you tap "Request Ride."  
The system needs to find all available drivers within, say, 3 km of you.

Each driver has a latitude and longitude. Your location is also a lat/lng pair.  
The question is: **which drivers are close to me?**

### Why Regular SQL Fails

A naive approach might be:
```sql
SELECT * FROM drivers 
WHERE ABS(lat - my_lat) < 0.03 
  AND ABS(lng - my_lng) < 0.03;
```

Problems:
1. **Inaccurate** — longitude degrees vary in distance depending on latitude
2. **Slow** — B-tree indexes can't efficiently search two columns at once
3. **No Earth curvature** — the Earth is round, distances aren't just Pythagoras

We need **spatial indexes** — data structures designed for multi-dimensional data.

## 🐌 Approach 1: Naive Python Scan (the bad baseline)

Before we reach for fancy spatial tools, let's try the obvious thing a beginner would write:

1. Fetch **all** drivers from the database.
2. Loop over them in Python and compute the distance to the rider.
3. Sort by distance, keep the closest 5.

This is the **O(N) per query** approach. It works fine for 10 drivers — and falls over at 5 million.
We compute distance with the **Haversine formula**, which accounts for the Earth's curvature.

In [ ]:
# Approach 1 — naive: scan every driver in Python
import math

def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance between two lat/lng points, in kilometers."""
    R = 6371.0  # Earth radius in km
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

rider_lat, rider_lng = 37.7749, -122.4194

conn = get_db(); cur = conn.cursor()
# Fetch EVERY driver (no spatial filter at the DB level — the bad part)
cur.execute("""
    SELECT d.id, d.name, d.status,
           ST_Y(dl.location::geometry) AS lat,
           ST_X(dl.location::geometry) AS lng
    FROM drivers d JOIN driver_locations dl ON d.id = dl.driver_id;
""")
rows = cur.fetchall()
conn.close()

# Compute distance in Python for each row, filter available, sort
scored = [
    (did, name, haversine_km(rider_lat, rider_lng, lat, lng))
    for did, name, status, lat, lng in rows
    if status == "available"
]
scored.sort(key=lambda x: x[2])

print(f"Scanned {len(rows)} drivers in Python, kept 5 nearest:")
for did, name, d in scored[:5]:
    print(f"  driver:{did:<3} {name:<16} {d:.2f} km")

print()
print("⚠️  Why this is the 'bad' baseline:")
print("   • Transfers every driver row over the wire on every request.")
print("   • Does the math in Python, not the database.")
print("   • Cost grows linearly with total drivers — at 5M drivers,")
print("     a single rider tap would read millions of rows. Not survivable.")
print("   Next: let the database prune by location using a spatial index.")

## 🗺️ Approach 2: PostGIS (Spatial Index)

PostGIS adds geographic data types to PostgreSQL. Instead of storing lat/lng as two numbers, we store a **GEOGRAPHY(POINT)** — a proper geospatial object.

PostGIS then uses a **GiST index** (Generalized Search Tree) which is like a B-tree but for shapes and regions. It divides space into bounding boxes, so "find all points within 3 km" only checks the relevant boxes.

Let's see it in action.

In [ ]:
# Let's see what drivers are in our database and where they are

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.status,
        d.vehicle_make || ' ' || d.vehicle_model AS vehicle,
        ST_Y(dl.location::geometry) AS latitude,
        ST_X(dl.location::geometry) AS longitude
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    ORDER BY d.id;
""")

print(f"{'ID':<4} {'Name':<16} {'Status':<12} {'Vehicle':<20} {'Lat':>10} {'Lng':>12}")
print("-" * 80)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<16} {row[2]:<12} {row[3]:<20} {row[4]:>10.4f} {row[5]:>12.4f}")

conn.close()

In [ ]:
# 🎯 Find the 5 nearest AVAILABLE drivers to a rider in downtown SF
#
# Rider location: downtown San Francisco (-122.4194, 37.7749)
# We use ST_DWithin to filter by distance, then ST_Distance to sort.

rider_lng = -122.4194
rider_lat = 37.7749
search_radius_meters = 5000  # 5 km

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.vehicle_make || ' ' || d.vehicle_model AS vehicle,
        ROUND(ST_Distance(
            dl.location,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        )::numeric, 0) AS distance_meters
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    WHERE d.status = 'available'
      AND ST_DWithin(
            dl.location,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            %s  -- radius in meters
          )
    ORDER BY distance_meters
    LIMIT 5;
""", (rider_lng, rider_lat, rider_lng, rider_lat, search_radius_meters))

print(f"🎯 Nearest available drivers to ({rider_lat}, {rider_lng}):")
print(f"   Search radius: {search_radius_meters / 1000} km")
print()
print(f"{'ID':<4} {'Name':<16} {'Vehicle':<20} {'Distance':>10}")
print("-" * 55)
for row in cur.fetchall():
    dist_km = float(row[3]) / 1000
    print(f"{row[0]:<4} {row[1]:<16} {row[2]:<20} {dist_km:>8.2f} km")

conn.close()

In [ ]:
# Let's see how the spatial index helps with EXPLAIN ANALYZE

conn = get_db()
cur = conn.cursor()

cur.execute("""
    EXPLAIN ANALYZE
    SELECT d.id, d.name
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    WHERE d.status = 'available'
      AND ST_DWithin(
            dl.location,
            ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography,
            5000
          );
""")

print("📊 Query Plan (notice the GiST index scan):")
print()
for row in cur.fetchall():
    print(f"  {row[0]}")

print()
print("💡 The GiST index lets PostGIS skip most of the table.")
print("   Without it, every row would need a distance calculation.")

conn.close()

## ⚡ Approach 3: Redis Geo (In-Memory Speed)

PostGIS is great for durable storage, but it lives on disk. With 2 million location updates per second, we need something faster.

**Redis Geo** stores locations in memory using **geohashing** — it converts (lat, lng) into a single integer that preserves spatial locality. Nearby points have similar geohash values, so Redis can find neighbors efficiently using its sorted set data structure.

Key commands:
- `GEOADD key lng lat member` — add/update a location
- `GEOSEARCH key FROMLONLAT lng lat BYRADIUS 5 km` — find nearby members
- `GEODIST key member1 member2 km` — distance between two members

In [ ]:
# Load all driver locations from PostGIS into Redis Geo
# In production, drivers send updates directly to Redis

r = get_redis()
conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.status,
        ST_X(dl.location::geometry) AS longitude,
        ST_Y(dl.location::geometry) AS latitude
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")

# Clear any existing data
r.delete("drivers:locations")
r.delete("drivers:available")

for row in cur.fetchall():
    driver_id, name, status, lng, lat = row
    
    # GEOADD adds the driver's location to a geo set
    r.geoadd("drivers:locations", (lng, lat, f"driver:{driver_id}"))
    
    # Track available drivers in a separate set for fast filtering
    if status == "available":
        r.sadd("drivers:available", f"driver:{driver_id}")
    
    print(f"  Added driver:{driver_id} ({name}) at ({lat:.4f}, {lng:.4f}) — {status}")

conn.close()
print()
print(f"✅ Loaded {r.zcard('drivers:locations')} drivers into Redis Geo")
print(f"   {r.scard('drivers:available')} are marked available")

In [ ]:
# 🎯 Find nearest drivers using Redis GEOSEARCH
# This is what happens in real time when a rider requests a ride

rider_lng = -122.4194
rider_lat = 37.7749
search_radius_km = 5

# GEOSEARCH returns members within radius, sorted by distance
nearby = r.geosearch(
    name="drivers:locations",
    longitude=rider_lng,
    latitude=rider_lat,
    radius=search_radius_km,
    unit="km",
    withcoord=True,
    withdist=True,
    sort="ASC",  # nearest first
    count=10
)

# Get the set of available drivers
available = r.smembers("drivers:available")

print(f"🎯 Redis GEOSEARCH: drivers within {search_radius_km} km of downtown SF")
print()
print(f"{'Driver':<14} {'Distance':>10} {'Available':>10} {'Coordinates':>24}")
print("-" * 62)
for member, dist, coords in nearby:
    is_available = "✅ yes" if member in available else "❌ no"
    print(f"{member:<14} {dist:>8.2f} km {is_available:>10}   ({coords[1]:.4f}, {coords[0]:.4f})")

# Filter to only available drivers
matches = [(m, d) for m, d, c in nearby if m in available]
print()
print(f"🚗 Best match: {matches[0][0]} at {matches[0][1]:.2f} km away" if matches else "❌ No available drivers nearby!")

In [ ]:
# ⏱️ Speed comparison: PostGIS vs Redis Geo

iterations = 100

# Measure PostGIS
conn = get_db()
postgis_times = []
for _ in range(iterations):
    cur = conn.cursor()
    start = time.time()
    cur.execute("""
        SELECT d.id
        FROM drivers d
        JOIN driver_locations dl ON d.id = dl.driver_id
        WHERE d.status = 'available'
          AND ST_DWithin(
                dl.location,
                ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography,
                5000)
        ORDER BY ST_Distance(
                dl.location,
                ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography)
        LIMIT 5;
    """)
    cur.fetchall()
    postgis_times.append((time.time() - start) * 1000)
conn.close()

# Measure Redis Geo
redis_times = []
for _ in range(iterations):
    start = time.time()
    r.geosearch(
        name="drivers:locations",
        longitude=-122.4194, latitude=37.7749,
        radius=5, unit="km",
        withdist=True, sort="ASC", count=5
    )
    redis_times.append((time.time() - start) * 1000)

avg_postgis = sum(postgis_times) / len(postgis_times)
avg_redis = sum(redis_times) / len(redis_times)

print(f"⏱️ Proximity Search Latency ({iterations} runs each):")
print(f"{'':>4}{'Avg':>10}{'Min':>10}{'Max':>10}")
print(f"  PostGIS: {avg_postgis:>8.2f}ms {min(postgis_times):>8.2f}ms {max(postgis_times):>8.2f}ms")
print(f"  Redis:   {avg_redis:>8.2f}ms {min(redis_times):>8.2f}ms {max(redis_times):>8.2f}ms")
print()
print(f"🚀 Redis is {avg_postgis / avg_redis:.1f}× faster for proximity searches!")
print()
print("💡 This is why Uber uses Redis for real-time matching")
print("   and PostGIS/similar for analytics and historical data.")

# The whole point of the Redis layer is that it wins this comparison. If it ever
# stops winning, the architecture argument in this notebook is wrong.
assert avg_redis < avg_postgis, (
    f"expected Redis GEOSEARCH to beat PostGIS for proximity search, got "
    f"redis={avg_redis:.2f}ms vs postgis={avg_postgis:.2f}ms"
)

## 🧠 How Geohashing Works (Behind Redis Geo)

Redis doesn't store raw lat/lng — it converts them into a **geohash**, a single integer that encodes both coordinates. Here's the key insight:

```
Geohashing divides the world into a grid of cells.
Each cell gets a unique code. The longer the code,
the smaller (more precise) the cell.

┌────────┬────────┐
│  00    │  01    │   2-bit geohash: 4 cells
├────────┼────────┤
│  10    │  11    │
└────────┴────────┘

┌────┬────┬────┬────┐
│0000│0001│0100│0101│
├────┼────┼────┼────┤  4-bit geohash: 16 cells
│0010│0011│0110│0111│
├────┼────┼────┼────┤
│1000│1001│1100│1101│
├────┼────┼────┼────┤
│1010│1011│1110│1111│
└────┴────┴────┴────┘
```

Points in the **same** cell share a common prefix.  
Redis stores geohashes in a sorted set, so everything in one cell lands in one contiguous range.

To find neighbours, Redis reads a handful of those ranges instead of scanning everything —
one range per cell it decides it has to check. *Which* cells those are turns out to be the
interesting part, and the next two cells are about exactly that.

In [ ]:
# Redis exposes the geohash string it derived for each member.
# Reference point: driver:2. For every driver we show its geohash, its true
# distance to driver:2, and how many leading characters the two geohashes share.

def shared_prefix(a, b):
    """Number of leading characters two geohash strings have in common."""
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n

ref = "driver:2"
ref_hash = r.geohash("drivers:locations", ref)[0]

print(f"{'Driver':<12} {'Geohash':<14} {'Dist to driver:2':>18} {'Shared prefix':>15}")
print("-" * 62)

rows = []
for driver_id in range(1, 6):
    member = f"driver:{driver_id}"
    gh = r.geohash("drivers:locations", member)[0]
    dist_km = float(r.geodist("drivers:locations", ref, member, unit="km"))
    shared = shared_prefix(ref_hash, gh)
    rows.append((member, gh, dist_km, shared))
    print(f"{member:<12} {gh:<14} {dist_km:>15.2f} km {shared:>15}")

others = [row for row in rows if row[0] != ref]
d3 = next(row for row in others if row[0] == "driver:3")
farther = [row for row in others if row[2] > d3[2]]

print()
print("⚠️  Read that table again. driver:3 is only "
      f"{d3[2]:.2f} km from driver:2 — closer than "
      f"{', '.join(row[0] for row in farther)} — yet it shares the FEWEST")
print(f"    geohash characters with driver:2 ({d3[3]} of 11). A shared prefix proves")
print("    two points are close. The reverse is not true.")

# This is the whole reason the next section exists. If the seed data ever moves
# so that driver:3 no longer straddles a cell boundary, this lesson is gone.
assert farther, "expected driver:4/driver:5 to be farther from driver:2 than driver:3 is"
assert all(row[3] > d3[3] for row in farther), (
    "expected the cell-boundary effect: driver:3 is nearer to driver:2 than "
    f"{[row[0] for row in farther]} yet shares fewer geohash characters. Got {rows}"
)

## 🕳️ The Cell-Boundary Trap

That asymmetry is the bug that eats almost every hand-rolled geohash index:

> **Same prefix ⇒ close. Close ⇏ same prefix.**

`driver:2` and `driver:3` are under 2 km apart, but a cell boundary runs between them,
so their geohashes diverge at the 4th character. A "find everyone whose geohash starts
with the same 5 characters" search will **never** return `driver:3` for a rider standing
next to `driver:2` — no matter how close the two get. And the same search happily returns
drivers in the far corner of the cell that are well outside the radius.

So prefix matching alone is wrong in *both* directions: it misses near points and it
returns far ones. The fix is what Redis (and S2, and H3) does internally:

1. Compute the cell containing the query point.
2. Also compute its **8 neighbouring cells** — 9 cells in total.
3. Union the members of all 9 cells to get a *candidate* set.
4. Apply an **exact distance filter** to the candidates.

Step 2 kills the false negatives. Step 4 kills the false positives. Neither step is
optional. Let's prove all of that against real data.

In [ ]:
# A minimal geohash implementation, so we can see exactly what prefix search does.
_B32 = "0123456789bcdefghjkmnpqrstuvwxyz"


def geohash_encode(lat, lng, precision):
    """Standard geohash: interleave lng/lat bisection bits, then base32 them."""
    lat_range, lng_range = [-90.0, 90.0], [-180.0, 180.0]
    out, bits, chunk, use_lng = [], 0, 0, True
    while len(out) < precision:
        rng = lng_range if use_lng else lat_range
        val = lng if use_lng else lat
        mid = (rng[0] + rng[1]) / 2
        if val > mid:
            chunk = (chunk << 1) | 1
            rng[0] = mid
        else:
            chunk = chunk << 1
            rng[1] = mid
        use_lng = not use_lng
        bits += 1
        if bits == 5:
            out.append(_B32[chunk])
            bits, chunk = 0, 0
    return "".join(out)


def geohash_bbox(h):
    """Decode a geohash back to the ([lat_lo, lat_hi], [lng_lo, lng_hi]) box it covers."""
    lat_range, lng_range = [-90.0, 90.0], [-180.0, 180.0]
    use_lng = True
    for c in h:
        v = _B32.index(c)
        for shift in range(4, -1, -1):
            bit = (v >> shift) & 1
            rng = lng_range if use_lng else lat_range
            mid = (rng[0] + rng[1]) / 2
            rng[0 if bit else 1] = mid
            use_lng = not use_lng
    return lat_range, lng_range


def geohash_neighbours(h):
    """The cell itself plus its 8 neighbours, at the same precision.

    We decode the cell to its bounding box, then re-encode the centre of each
    adjacent box. That lands squarely inside the neighbour, so it is exact --
    no fudge factors.
    """
    (lat_lo, lat_hi), (lng_lo, lng_hi) = geohash_bbox(h)
    clat, clng = (lat_lo + lat_hi) / 2, (lng_lo + lng_hi) / 2
    dh, dw = lat_hi - lat_lo, lng_hi - lng_lo
    return {
        geohash_encode(clat + dy * dh, clng + dx * dw, len(h))
        for dy in (-1, 0, 1)
        for dx in (-1, 0, 1)
    }


PRECISION = 5      # ~4.9 km x 4.9 km cells -- same order as our search radius
RADIUS_KM = 3.0

# Rider standing right next to driver:2.
rider_lng, rider_lat = -122.4089, 37.7837

# Read every driver's stored position straight out of Redis.
members = sorted(r.zrange("drivers:locations", 0, -1), key=lambda m: int(m.split(":")[1]))
positions = {m: r.geopos("drivers:locations", m)[0] for m in members}

rider_cell = geohash_encode(rider_lat, rider_lng, PRECISION)
nine_cells = geohash_neighbours(rider_cell)


def cell_of(member):
    lng, lat = positions[member]
    return geohash_encode(lat, lng, PRECISION)


def dist_of(member):
    lng, lat = positions[member]
    return haversine_km(rider_lat, rider_lng, lat, lng)


# ❌ Naive: "same cell means nearby"
naive = {m for m in members if cell_of(m) == rider_cell}
# ⚠️  Steps 1-3 only: the 9-cell candidate set, no distance filter yet
candidates = {m for m in members if cell_of(m) in nine_cells}
# ✅ Steps 1-4: candidates cut down by true distance
correct = {m for m in candidates if dist_of(m) <= RADIUS_KM}
# Ground truth: let Redis answer the same question
truth = {
    m for m, _ in r.geosearch(
        name="drivers:locations",
        longitude=rider_lng, latitude=rider_lat,
        radius=RADIUS_KM, unit="km", withdist=True,
    )
}

print(f"Rider cell (precision {PRECISION}): {rider_cell}")
print(f"9-cell search set: {sorted(nine_cells)}")
print()
print(f"{'Driver':<12} {'Cell':<8} {'Distance':>10}   {'naive':<7}{'9-cell':<8}{'9-cell+dist':<13}{'GEOSEARCH'}")
print("-" * 78)
for m in members:
    tick = lambda s: "  ✅   " if m in s else "  ·    "
    print(f"{m:<12} {cell_of(m):<8} {dist_of(m):>7.2f} km  "
          f"{tick(naive):<7}{tick(candidates):<8}{tick(correct):<13}{tick(truth)}")

missed = truth - naive
overshoot = candidates - truth
print()
print(f"❌ Same-cell prefix search MISSED {sorted(missed)} — genuinely within {RADIUS_KM} km.")
print(f"⚠️  The raw 9-cell candidate set OVERSHOT with {sorted(overshoot)} — outside the radius.")
print(f"✅ 9 cells + exact distance filter == GEOSEARCH: {sorted(correct)}")

# Each assertion pins down one half of the lesson.
assert missed, (
    "prefix-only search was supposed to miss a nearby driver; if it did not, the "
    "seed data no longer straddles a cell boundary and this demo teaches nothing"
)
assert "driver:3" in missed, f"expected driver:3 to be the false negative, got {sorted(missed)}"
assert overshoot, (
    "the unfiltered candidate set was supposed to overshoot the radius; without "
    "false positives there is nothing for step 4 to fix"
)
assert correct == truth, (
    f"9 cells + exact filter must reproduce GEOSEARCH exactly; differ by {sorted(correct ^ truth)}"
)

## 🏗️ Building the Match Function

Let's combine everything into a function that simulates what happens when a rider requests a ride. The matching algorithm:

1. Use Redis GEOSEARCH to find nearby drivers (fast)
2. Filter to only available drivers
3. Return the ranked list (closest first)

In [ ]:
def find_nearby_drivers(rider_lng, rider_lat, radius_km=5, max_results=5):
    """
    Find the nearest available drivers to a rider's location.
    Uses Redis Geo for fast proximity search.
    
    Returns a list of (driver_key, distance_km) tuples.
    """
    r = get_redis()
    
    # Step 1: GEOSEARCH for drivers within radius
    nearby = r.geosearch(
        name="drivers:locations",
        longitude=rider_lng,
        latitude=rider_lat,
        radius=radius_km,
        unit="km",
        withdist=True,
        sort="ASC",
        count=max_results * 3  # fetch extra to account for filtering
    )
    
    # Step 2: filter to only available drivers
    available = r.smembers("drivers:available")
    matches = [
        (member, float(dist))
        for member, dist in nearby
        if member in available
    ]
    
    # Step 3: return top matches (already sorted by distance)
    return matches[:max_results]


# Test it!
print("🚗 Scenario 1: Rider in Downtown SF")
matches = find_nearby_drivers(-122.4194, 37.7749)
for driver, dist in matches:
    print(f"   {driver}: {dist:.2f} km away")

print()
print("🚗 Scenario 2: Rider near Outer Sunset (fewer drivers)")
matches = find_nearby_drivers(-122.4900, 37.7550, radius_km=3)
for driver, dist in matches:
    print(f"   {driver}: {dist:.2f} km away")
if not matches:
    print("   ❌ No drivers found! Expanding search radius...")
    matches = find_nearby_drivers(-122.4900, 37.7550, radius_km=8)
    for driver, dist in matches:
        print(f"   {driver}: {dist:.2f} km away")

# 🔎 Cross-check the two approaches against each other.
# The naive haversine scan from Approach 1 and the Redis geo index are computing
# the same thing two different ways. If they ever disagree, something is broken:
# a swapped lat/lng, a wrong radius unit, or a flat-Earth distance formula.
print()
print("🔎 Cross-check: Redis GEOSEARCH vs the naive haversine scan")
redis_top3 = find_nearby_drivers(-122.4194, 37.7749, radius_km=5, max_results=3)
naive_top3 = [(f"driver:{did}", dist) for did, _, dist in scored[:3]]
for (member, redis_km), (_, naive_km) in zip(redis_top3, naive_top3):
    print(f"   {member}: redis={redis_km:.3f} km  haversine={naive_km:.3f} km")

assert [m for m, _ in redis_top3] == [m for m, _ in naive_top3], (
    f"Redis and the naive scan disagree on ranking: {redis_top3} vs {naive_top3}"
)
for (member, redis_km), (_, naive_km) in zip(redis_top3, naive_top3):
    # Redis uses a slightly larger Earth radius and quantises coordinates to
    # ~0.6 m, so allow 20 m or 1%, whichever is bigger.
    assert abs(redis_km - naive_km) <= max(0.02, 0.01 * naive_km), (
        f"{member}: Redis says {redis_km:.3f} km, haversine says {naive_km:.3f} km"
    )
print("   ✅ Same ranking, same distances.")

## 🧹 Cleanup

In [ ]:
r = get_redis()
r.delete("drivers:locations", "drivers:available")
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **Regular indexes can't do proximity search** — B-trees work for one dimension, not two
2. **PostGIS** adds spatial indexes (GiST) to PostgreSQL — great for durable storage and analytics
3. **Redis Geo** uses geohashing for in-memory proximity searches — measurably faster than
   PostGIS here (see the timing cell for the number on *your* machine; the gap widens in
   production, where PostGIS also pays disk, WAL and MVCC costs that Redis does not)
4. **Uber uses both** — Redis for real-time matching, PostGIS/similar for historical data
5. **GEOSEARCH** is the key Redis command — finds nearby members by radius, sorted by distance
6. **Never hand-roll prefix matching** — a shared geohash prefix proves two points are close,
   but two points metres apart can land in different cells. A correct search reads the query
   cell *plus its 8 neighbours*, then applies an exact distance filter. Redis, S2 and H3 all
   do this for you; a `LIKE 'geohash%'` query does not.

### How This Fits in a System Design Interview

When asked "how do you find nearby drivers?", the progression is:
- ❌ Naive: scan all drivers and calculate distances → O(n) per request
- ✅ Good: PostGIS with spatial index → efficient disk-based proximity search  
- ✅ Great: Redis Geo → in-memory, handles 2M updates/sec + fast proximity search

And the follow-up the interviewer is waiting for: *"what about points near a cell boundary?"*

### Next Up

In **Notebook 2**, we'll tackle **real-time driver tracking** — how to handle millions of location updates per second and keep the geo index fresh.